# 23 - LoFTR sobre SB013 segmentado

Este notebook repite la primera prueba deep de matching realizada con LoFTR, pero incorpora las segmentaciones H&E y HSI desarrolladas posteriormente.

La pregunta es sencilla: **si eliminamos la caja, el fondo y las regiones que no pertenecen al especimen, mejora el matching de LoFTR?**

## Donde encaja este experimento

LoFTR no es una red de registration deformable como VoxelMorph o TransMorph. Es un matcher de caracteristicas detector-free basado en Transformers y preentrenado sobre imagenes naturales.

En esta prueba:

1. LoFTR encuentra correspondencias entre la HSI pseudo-RGB y la H&E.
2. RANSAC elimina correspondencias inconsistentes.
3. Se estima una transformacion affine y una homografia.
4. La HSI se transforma al sistema de referencia de la H&E, igual que en `1_Prueba_Deep.ipynb`.

No se entrena LoFTR con nuestros seis sujetos.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display

ROOT = Path.cwd()
METHOD_DIR = ROOT / 'Registration' / 'DeepLearning' / 'Tecnicas_registration' / '05_feature_based_matching'
SCRIPT = METHOD_DIR / 'scripts' / 'run_loftr_sb013_segmented.py'
OUTPUT_DIR = METHOD_DIR / 'outputs' / 'outputs_loftr_SB013_segmented'
CSV_PATH = OUTPUT_DIR / 'loftr_SB013_comparison_summary.csv'
JSON_PATH = OUTPUT_DIR / 'loftr_SB013_comparison_summary.json'

print('Script:', SCRIPT)
print('Outputs:', OUTPUT_DIR)

## Casos comparados

- `original_full`: HSI pseudo-RGB completa y macrofotografia H&E completa, reproduciendo el planteamiento inicial.
- `scaled_unmasked`: imagenes preparadas a escala fisica comparable, antes del recorte exacto por mascara.
- `segmented_cropped`: solo el especimen segmentado, con fondo negro y recorte al bounding box de cada mascara.

Se mantiene `KF.LoFTR(pretrained="outdoor")` para no cambiar el matcher respecto a la prueba original.

In [ ]:
# Cambia a True si quieres volver a ejecutar LoFTR y regenerar todos los outputs.
RUN_EXPERIMENT = False

if RUN_EXPERIMENT:
    result = subprocess.run([sys.executable, str(SCRIPT)], cwd=str(ROOT), text=True, capture_output=True)
    print(result.stdout[-8000:])
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError('Fallo el experimento LoFTR segmentado')

In [ ]:
metrics = pd.read_csv(CSV_PATH)
display(metrics.round(4))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 13))
axes[0, 0].imshow(Image.open(OUTPUT_DIR / 'original_full_inputs.png'))
axes[0, 0].set_title('Entradas originales completas')
axes[0, 1].imshow(Image.open(OUTPUT_DIR / 'segmented_cropped_inputs.png'))
axes[0, 1].set_title('Entradas segmentadas y recortadas')
axes[1, 0].imshow(Image.open(OUTPUT_DIR / 'original_full_loftr_matches.png'))
axes[1, 0].set_title('Matches LoFTR originales')
axes[1, 1].imshow(Image.open(OUTPUT_DIR / 'segmented_cropped_loftr_matches.png'))
axes[1, 1].set_title('Matches LoFTR con segmentacion')
for ax in axes.ravel():
    ax.axis('off')
plt.tight_layout()

In [ ]:
plt.figure(figsize=(16, 5))
plt.imshow(Image.open(OUTPUT_DIR / 'loftr_SB013_comparison_summary.png'))
plt.axis('off')
plt.title('Comparacion cuantitativa de las tres variantes')
plt.tight_layout()

## Resultado principal

La segmentacion produce mas correspondencias:

- Prueba original: **55 matches**.
- Imagenes preparadas sin recorte explicito: **114 matches**.
- Especimen segmentado y recortado: **142 matches**.

Para la homografia, el Dice de mascara pasa de **0.838** a **0.903**, una mejora de **+0.065**.

Sin embargo, el ratio de inliers de la homografia baja de **0.202** a **0.134**. Esto significa que aparecen mas matches, pero una proporcion mayor es inconsistente y debe ser rechazada por RANSAC.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(Image.open(OUTPUT_DIR / 'scaled_unmasked_homography_overlay.png'))
axes[0].set_title('Sin recorte explicito | Homography')
axes[1].imshow(Image.open(OUTPUT_DIR / 'segmented_cropped_affine_overlay.png'))
axes[1].set_title('Segmentado | Affine')
axes[2].imshow(Image.open(OUTPUT_DIR / 'segmented_cropped_homography_overlay.png'))
axes[2].set_title('Segmentado | Homography')
for ax in axes:
    ax.axis('off')
plt.tight_layout()

## Lectura critica

La segmentacion **si ayuda a LoFTR**, especialmente cuando las correspondencias se utilizan para estimar una homografia:

- El numero de matches aumenta de 55 a 142.
- El Dice segmentado con homografia alcanza 0.903.
- La transformacion affine segmentada queda en 0.764, por lo que el beneficio no aparece en todos los modelos geometricos.
- Los matches siguen siendo ruidosos y se concentran principalmente en bordes y estructuras de contraste fuerte.
- El baseline clasico de SB013 alcanza aproximadamente 0.984 de Dice, por encima de LoFTR + homografia.

**Conclusion:** eliminar caja y fondo mejora la utilidad de LoFTR como inicializacion feature-based, pero LoFTR por si solo no sustituye al pipeline de registration. La version segmentada puede presentarse como una mejora clara respecto a la primera prueba exploratoria.